# Refined detector evaluation

Saved best-checkpoint metrics and cached validation predictions. This notebook performs no model training or model inference. The refinement uses additional compute; the YOLO row is an HTR alternative, not an equal-budget final-model comparison.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import json
import hashlib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.patches import Rectangle, Patch
from PIL import Image
from cv import CLASSES, pages_with_split, subset, suppress_duplicates, match_boxes
from htr import snapshot, is_htr_line, clean_text, REVISION, SPLIT_FILE

RESULTS = ROOT / 'results/cv'
bench = json.loads((RESULTS / 'benchmark/runs.json').read_text())
refined = json.loads((RESULTS / 'rf-small-refined.json').read_text())['rf-small-refined']
assert refined['training']['dataset_revision'] == REVISION
assert refined['training']['split_sha256'] == hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest()
records = [('RF-DETR Small', bench['rf-small'], '50 min'),
           ('RF-DETR Small refined', refined, '50 + 120 min'),
           ('YOLO26 Small', bench['yolo26s'], '50 min')]
rows = []
for name, record, time in records:
    m = record['metrics']
    rows.append(dict(model=name, training=time, mAP=m['raw']['mAP'], AP50=m['raw']['AP50'],
                     handwritten_AP=m['raw']['AP_by_type']['handwritten'], F1=m['filtered']['detection_F1'],
                     coverage=m['handwritten_coverage']['matched_coverage']))
comparison = pd.DataFrame(rows)
comparison.round(4)

,model,training,mAP,AP50,handwritten_AP,F1,coverage
0,RF-DETR Small,50 min,0.4251,0.6475,0.5441,0.8789,0.8661
1,RF-DETR Small refined,50 + 120 min,0.5105,0.7310,0.5543,0.9104,0.8969
2,YOLO26 Small,50 min,0.2499,0.4454,0.5311,0.8924,0.9008


In [2]:
val = subset(pages_with_split(), 'val')
counts = pd.Series([r['type'] for p in val for r in p['regions']]).value_counts()
by_type = pd.DataFrame({'GT regions': counts,
    'benchmark AP': bench['rf-small']['metrics']['raw']['AP_by_type'],
    'refined AP': refined['metrics']['raw']['AP_by_type']}).reindex(CLASSES)
by_type['change'] = by_type['refined AP'] - by_type['benchmark AP']
by_type.round(4)

,GT regions,benchmark AP,refined AP,change
handwritten,2107,0.5441,0.5543,0.0102
printed,4,0.1048,0.1396,0.0348
formula,262,0.4974,0.5004,0.0030
table,10,0.5662,0.6076,0.0414
annotation,68,0.2601,0.3100,0.0499
image,8,0.4084,0.6725,0.2641
graph,4,0.5944,0.7891,0.1947


In [3]:
source_scores = pd.DataFrame(refined['metrics']['per_source']).T
source_scores[['pages', 'detection_precision', 'detection_recall', 'detection_F1']].round(3)

,pages,detection_precision,detection_recall,detection_F1
archive,13,0.977346,0.937888,0.957211
dictation,36,0.968872,0.923362,0.94557
school,68,0.897959,0.864629,0.880979
university,16,0.919571,0.824519,0.869455


In [4]:
coverage = pd.DataFrame({name: record['metrics']['handwritten_coverage'] for name, record, _ in records})
coverage.loc[['eligible_gt_lines', 'matched_lines', 'missed_lines', 'unmatched_predictions', 'matched_coverage']]

,RF-DETR Small,RF-DETR Small refined,YOLO26 Small
eligible_gt_lines,2076.000000,2076.000000,2076.000000
matched_lines,1798.000000,1862.000000,1870.000000
missed_lines,278.000000,214.000000,206.000000
unmatched_predictions,215.000000,201.000000,252.000000
matched_coverage,0.866089,0.896917,0.900771


In [5]:
history = pd.DataFrame(refined['training']['history'])
best = history.loc[history.mAP.idxmax()]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history.epoch, history.mAP, '.-', label='Refinement')
axes[0].axhline(bench['rf-small']['metrics']['raw']['mAP'], color='gray', linestyle='--', label='50-minute checkpoint')
axes[0].scatter([best.epoch], [best.mAP], color='green', s=65, zorder=3, label=f'Best: epoch {int(best.epoch)}')
axes[0].set(xlabel='Refinement epoch', ylabel='Validation mAP', title='Best checkpoint retained at early stopping')
axes[0].legend(fontsize=8)
by_type[['benchmark AP', 'refined AP']].plot.bar(ax=axes[1], rot=35)
axes[1].set_xticklabels([f'{kind} (n={int(counts[kind])})' for kind in CLASSES], ha='right')
axes[1].set(ylabel='Raw AP', title='Per-class AP and validation support', ylim=(0, 1))
fig.tight_layout()
fig.savefig(RESULTS / 'rf-small-refined_metrics.png', dpi=140)
plt.close(fig)

![Learning curve and per-class AP](../results/cv/rf-small-refined_metrics.png)

Most of the overall mAP increase comes from image and graph AP. These classes have only eight and four validation regions, respectively. Handwritten AP and crop coverage are more direct indicators for HTR integration.

In [6]:
CACHE = ROOT / 'models/rf-small-refined/val_predictions.json'
THRESHOLD = refined['metrics']['filtered']['score_threshold']
if CACHE.exists():
    raw = json.loads(CACHE.read_text())
    predictions = {p['file_name']: p for p in suppress_duplicates(raw, refined['metrics']['nms_iou'])}
    page_scores = []
    missed_by_page = {}
    for page in val:
        pred = predictions[page['file_name']]
        eligible = [(i, r) for i, r in enumerate(page['regions']) if is_htr_line(r) and clean_text(r['text'])]
        boxes = [b for b, label, score in zip(pred['boxes'], pred['labels'], pred['scores'])
                 if label == 0 and score >= THRESHOLD]
        matches = match_boxes(boxes, [r['bbox'] for _, r in eligible])
        matched_ids = {eligible[gi][0] for _, gi, _ in matches}
        missed_by_page[page['file_name']] = {i for i, _ in eligible} - matched_ids
        page_scores.append(dict(file_name=page['file_name'], source=page['source'], eligible=len(eligible),
            matched=len(matches), missed=len(eligible)-len(matches), unmatched=len(boxes)-len(matches),
            coverage=len(matches)/len(eligible) if eligible else np.nan))
    page_scores = pd.DataFrame(page_scores)
    assert int(page_scores.matched.sum()) == refined['metrics']['handwritten_coverage']['matched_lines']
    assert int(page_scores.unmatched.sum()) == refined['metrics']['handwritten_coverage']['unmatched_predictions']

In [7]:
def plot_examples(chosen, filename, failures=False):
    colors = {kind: f'C{i}' for i, kind in enumerate(CLASSES)}
    fig, axes = plt.subplots(len(chosen), 2, figsize=(11, 5*len(chosen)), squeeze=False)
    for row, page in zip(axes, chosen):
        pred = predictions[page['file_name']]
        with Image.open(snapshot() / 'train' / page['file_name']) as original:
            width, height = original.size
            image = original.convert('RGB')
            image.thumbnail((700, 850))
        truth = [(r['bbox'], r['type'], i) for i, r in enumerate(page['regions'])]
        detected = [(b, CLASSES[label], None) for b, label, score in zip(pred['boxes'], pred['labels'], pred['scores']) if score >= THRESHOLD]
        for axis, regions, title in zip(row, (truth, detected), ('Ground truth', 'Refined predictions')):
            axis.imshow(image)
            for (x1, y1, x2, y2), kind, index in regions:
                color = ('red' if index in missed_by_page[page['file_name']] else 'gray') if failures and title == 'Ground truth' else colors[kind]
                axis.add_patch(Rectangle((x1*image.width/width, y1*image.height/height),
                    (x2-x1)*image.width/width, (y2-y1)*image.height/height, fill=False, edgecolor=color, linewidth=.9))
            axis.set_title(f'{page["source"]}: {title} ({len(regions)})')
            axis.axis('off')
    fig.legend(handles=[Patch(color=colors[kind], label=kind) for kind in CLASSES], loc='lower center', ncol=4)
    fig.tight_layout(rect=(0, .04, 1, 1))
    fig.savefig(RESULTS / filename, dpi=110)
    plt.close(fig)

if CACHE.exists():
    chosen = [next(p for p in val if p['source'] == source) for source in ('dictation', 'archive', 'university', 'school')]
    plot_examples(chosen, 'rf-small-refined_examples.png')
    worst = page_scores[page_scores.eligible >= 10].sort_values(['coverage', 'file_name']).head(2)
    chosen = [next(p for p in val if p['file_name'] == name) for name in worst.file_name]
    plot_examples(chosen, 'rf-small-refined_failures.png', failures=True)
    display(worst[['source', 'eligible', 'matched', 'missed', 'coverage']].round(3))

,source,eligible,matched,missed,coverage
74,school,10,2,8,0.200
88,school,44,13,31,0.295


![Representative validation pages](../results/cv/rf-small-refined_examples.png)

![Failure cases](../results/cv/rf-small-refined_failures.png)

Failure cases are the two lowest handwritten-coverage pages with at least ten eligible lines. Red GT boxes have no matched handwritten prediction at IoU 0.5; other GT boxes are gray. Splits across columns and merges into a table also count as misses. Coverage and matched-crop recognition scores are distinct from full-page text accuracy.

In [8]:
summary = ['# RF-DETR Small refinement', '',
    'Early stopping after 17 epochs (120 additional minutes); best checkpoint: epoch 12. Same split and 640-pixel input, 30% learning rates.', '',
    '| Run | Training | mAP | AP50 | Handwritten AP | Box F1 | Coverage |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: |']
for row in comparison.itertuples():
    summary.append(f'| {row.model} | {row.training} | {row.mAP:.4f} | {row.AP50:.4f} | {row.handwritten_AP:.4f} | {row.F1:.4f} | {row.coverage:.1%} |')
summary += ['', 'Refinement recovers 64 more eligible handwritten lines than its benchmark checkpoint, with unmatched predictions reduced from 215 to 201. YOLO recovers eight more lines than the refinement, with 51 more unmatched predictions.', '',
    'About 77% of the mAP gain comes from image/graph AP (only eight/four validation regions). The handwritten AP gain is 0.0102; coverage rises by 3.1 percentage points. One seed and validation-tuned filtering; official test is unused. Failure examples include cross-column splits and a diagram predicted as a table.', '',
    '![Learning curve and class AP](rf-small-refined_metrics.png)', '',
    '![Representative pages](rf-small-refined_examples.png)', '',
    '![Failure cases: unmatched eligible lines in red on GT panels](rf-small-refined_failures.png)']
_ = (RESULTS / 'rf-small-refined.md').write_text('\n'.join(summary) + '\n')